# Drug Repurposing Walkthrough — OptimShortestPaths.jl

This notebook reproduces every key result from the paper:

**"Interpretable Multi-Objective Drug Repurposing via Shortest-Path Casting on Biomedical Knowledge Graphs"**

*Tianchi Chen & Shihua Yu*

---

**Data sources (all real, zero mock):**
- Hetionet v1.0 (47,031 nodes, 2.25M edges)
- PharmacotherapyDB v1.0 (755 disease-modifying indications)
- SIDER 4.1 (label-derived adverse event frequencies)
- Precomputed results from `ChemPath/data/paper_results/`

**Requirements:** `pip install matplotlib numpy` (in the ChemPath venv)

## 1. Load Precomputed Results

All numbers were computed by `paper_experiments.py` and `kge_baseline.py` on real data.
We load the JSON files — no recomputation needed.

In [ ]:
import json
from pathlib import Path

RESULTS = Path("../ChemPath/data/paper_results")

with open(RESULTS / "paper_experiments.json") as f:
    exp = json.load(f)
with open(RESULTS / "kge_baseline.json") as f:
    kge = json.load(f)

print("Loaded experiment results.")
print(f"Methods: {list(exp['method_comparison'].keys())}")
print(f"KGE baselines: {list(kge.keys())}")

## 2. Table 3 — AUROC + AUPRC (all methods)

Corresponds to Table 3 in the paper. Every number printed below is
read directly from the JSON (traceable to `compute_auroc()` output).

In [ ]:
print(f"{'Method':<30s} {'AUROC':>8s} {'95% CI':>18s} {'AUPRC':>8s}")
print(f"{'-'*30} {'-'*8} {'-'*18} {'-'*8}")

labels = {
    'eff_evd_fused': 'Ours: Eff+Evidence',
    'efficacy_1d': 'Ours: Efficacy 1D',
    'eff_evd_safety_fused': 'Ours: Eff+Evd+Safety',
    'weighted_3d': 'Ours: Weighted 3D',
    'eff_safety_2d': 'Ours: Eff+Safety 2D',
}

for key, label in labels.items():
    d = exp['method_comparison'][key]
    print(f"{label:<30s} {d['auroc']:>8.4f} [{d['ci_lo']:.4f}, {d['ci_hi']:.4f}] {d['auprc']:>8.4f}")

print()
for model in ['ComplEx', 'TransE']:
    d = kge[model]
    print(f"{'KGE: ' + model:<30s} {d['auroc']:>8.4f} {'---':>18s} {d['auprc']:>8.4f}")

## 3. Hits@K

Fraction of diseases where at least one true positive appears in top-K.

In [ ]:
print(f"{'Method':<25s} {'H@10':>8s} {'H@50':>8s} {'H@100':>8s}")
print(f"{'-'*25} {'-'*8} {'-'*8} {'-'*8}")
for m, label in [('efficacy_1d', 'Efficacy 1D'), ('eff_safety_2d', 'Eff+Safety 2D')]:
    hk = exp['hits_at_k'][m]
    print(f"{label:<25s} {hk['10']:>8.3f} {hk['50']:>8.3f} {hk['100']:>8.3f}")

## 4. Path Length Analysis

Validates the core hypothesis: true drug-disease pairs have shorter paths.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

path = exp['path_length_analysis']

print(f"Positive pairs: n={path['n_positive']}, mean={path['pos_mean']:.3f}, median={path['pos_median']:.3f}")
print(f"Negative pairs: n={path['n_negative']}, mean={path['neg_mean']:.3f}, median={path['neg_median']:.3f}")
print(f"Cohen's d = {path['cohens_d']:.3f} ({path['separation']})")
print(f"KS statistic = {path['ks_statistic']:.3f}")

# Reproduce Figure 3 from paper
bins = path['bin_edges']
pos_density = np.array(path['pos_hist']) / path['n_positive']
neg_density = np.array(path['neg_hist']) / path['n_negative']
centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins)-1)]
w = (bins[1] - bins[0]) * 0.4

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([c - w/2 for c in centers], pos_density, width=w, color='#2E86AB',
       edgecolor='black', linewidth=0.4, label=f'Positive (n={path["n_positive"]})', alpha=0.85)
ax.bar([c + w/2 for c in centers], neg_density, width=w, color='#A23B72',
       edgecolor='black', linewidth=0.4, label=f'Negative (n={path["n_negative"]})', alpha=0.85)
ax.axvline(path['pos_mean'], color='#2E86AB', ls='--', lw=1.5, alpha=0.7)
ax.axvline(path['neg_mean'], color='#A23B72', ls='--', lw=1.5, alpha=0.7)
ax.set_xlabel('Shortest-path efficacy distance')
ax.set_ylabel('Density (within class)')
ax.set_title(f"Cohen's d = {path['cohens_d']:.2f}, KS = {path['ks_statistic']:.3f}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 5. Five-Fold Cross-Validation

Disease-level split, 5 folds. Confirms stability across random splits.

In [ ]:
cv = exp['cross_validation']

print(f"{'Method':<25s} {'Mean AUROC':>11s} {'Std':>8s} {'Per-fold':>40s}")
print(f"{'-'*25} {'-'*11} {'-'*8} {'-'*40}")
for m, label in [('efficacy_1d', 'Efficacy 1D'), ('weighted_3d', 'Weighted 3D'), ('eff_safety_2d', 'Eff+Safety 2D')]:
    d = cv[m]
    folds_str = ', '.join(f"{f:.4f}" for f in d['folds'])
    print(f"{label:<25s} {d['mean']:>11.4f} {d['std']:>8.4f} [{folds_str}]")

## 6. Sensitivity Analysis

AUROC vs p_base(binds). Spread < 0.02 = robust.

In [ ]:
sens = exp['pbase_sensitivity']
p_vals = [r['p_binds'] for r in sens]
aurocs = [r['auroc_1d'] for r in sens]
spread = max(aurocs) - min(aurocs)

print(f"{'p_binds':>8s} {'AUROC':>8s} {'AUPRC':>8s}")
print(f"{'-'*8} {'-'*8} {'-'*8}")
for r in sens:
    marker = ' ← default' if r['p_binds'] == 0.8 else ''
    print(f"{r['p_binds']:>8.2f} {r['auroc_1d']:>8.4f} {r['auprc_1d']:>8.4f}{marker}")

print(f"\nSpread: {spread:.4f} → {'ROBUST' if spread < 0.02 else 'SENSITIVE'}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(p_vals, aurocs, 'o-', color='#2E86AB', linewidth=2, markersize=8)
ax.axhline(np.mean(aurocs), color='gray', ls=':', lw=0.8)
ax.set_xlabel('p_base(binds)')
ax.set_ylabel('AUROC (1D)')
ax.set_title(f'Sensitivity: spread = {spread:.4f}')
plt.tight_layout()
plt.show()

## 7. Systematic Pareto Rescue

Aggregate results across all 137 diseases.

In [ ]:
pr = exp['pareto_rescue']

print(f"Total diseases analyzed:            {pr['total_diseases']}")
print(f"Diseases with any rescue:           {pr['diseases_with_any_rescue']} ({100*pr['diseases_with_any_rescue']/pr['total_diseases']:.1f}%)")
print(f"Diseases with validated rescue:     {pr['diseases_with_validated_rescue']}")
print(f"Total rescued candidates:           {pr['total_rescued_candidates']}")
print(f"Total validated rescues:            {pr['total_validated_rescues']}")

if pr['validated_rescue_details']:
    print(f"\nTop 10 validated rescues (by rank improvement):")
    print(f"{'Drug':<20s} {'Disease':<25s} {'1D→Pareto':>10s} {'Δ':>5s}")
    print(f"{'-'*20} {'-'*25} {'-'*10} {'-'*5}")
    for r in pr['validated_rescue_details'][:10]:
        drug = r['drug'].replace('Compound::', '')
        disease = r['disease'].replace('Disease::', '')
        print(f"{drug:<20s} {disease:<25s} {r['eff_rank']:>4}→{r['pareto_rank']:<4} +{r['delta']}")

## 8. Method Comparison Bar Chart

Reproduces Figure 2 from the paper.

In [ ]:
methods_order = [
    ('eff_evd_fused', 'Eff+Evidence'),
    ('efficacy_1d', 'Efficacy 1D'),
    ('eff_evd_safety_fused', 'Eff+Evd+Safety'),
    ('weighted_3d', 'Weighted 3D'),
    ('eff_safety_2d', 'Eff+Safety 2D'),
]
names = [l for _, l in methods_order] + ['KGE: ComplEx', 'KGE: TransE']
auroc_vals = [exp['method_comparison'][k]['auroc'] for k, _ in methods_order] + \
             [kge['ComplEx']['auroc'], kge['TransE']['auroc']]
colors = ['#2E86AB'] * 5 + ['#A23B72'] * 2

fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(names))
ax.barh(y, auroc_vals, color=colors, edgecolor='black', linewidth=0.4, alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels(names)
ax.invert_yaxis()
ax.set_xlabel('AUROC')
ax.set_xlim(0.5, 0.85)
ax.axvline(0.5, color='gray', ls=':', lw=0.8)
for i, v in enumerate(auroc_vals):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)
ax.set_title('AUROC: Our methods (blue) vs KGE baselines (magenta)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 9. Summary

All numbers above are traced to `paper_experiments.json` and `kge_baseline.json`,
which were produced by running the full pipeline on:
- Hetionet v1.0 (47,031 nodes, 869,528 mechanistic edges)
- PharmacotherapyDB (755 DM indications, 53,019 evaluation pairs)
- SIDER 4.1 (515 compounds with label frequencies + 561 fallback)
- PyKEEN (TransE + ComplEx, 100 epochs, 128-dim, CPU)

**No mock data was used at any point in this notebook.**